# 04 — Retrospective 2026 World Cup validation

This notebook is deliberately downstream of model development. It loads the selected probability model from disk and does not retrain, retune or reselect it after tournament outcomes are introduced.

All 104 World Cup fixtures are evaluated from the same frozen team-state snapshot ending on 31 March 2026. The analysis leads with log loss, multiclass Brier score, draw-probability behaviour and bootstrap uncertainty; hard-class diagnostics are secondary.

In [ ]:
from pathlib import Path
import json
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    log_loss,
    recall_score,
)

current_dir = Path.cwd()
if (current_dir / "data" / "results.csv").exists():
    project_root = current_dir
elif (current_dir.parent / "data" / "results.csv").exists():
    project_root = current_dir.parent
else:
    raise FileNotFoundError("Could not locate the project root.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.worldcup_runtime import (
    CLASS_LABELS,
    CLASS_NAMES,
    MatchPredictor,
    align_probability_columns,
    multiclass_brier_score,
    run_monte_carlo_tournament,
)

outputs_dir = project_root / "outputs"
models_dir = project_root / "models"

selected_probability_model = joblib.load(
    models_dir / "selected_probability_model.joblib"
)
dummy_model = joblib.load(
    models_dir / "class_prior_baseline.joblib"
)
model_metadata = json.loads(
    (models_dir / "model_metadata.json").read_text(
        encoding="utf-8"
    )
)
team_state_snapshot = pd.read_csv(
    outputs_dir / "team_state_snapshot.csv",
    parse_dates=["latest_match_date"],
)

predictor = MatchPredictor(
    selected_probability_model,
    team_state_snapshot,
    feature_columns=model_metadata["feature_columns"],
)

world_cup_actual = pd.read_csv(
    project_root / "data" / "world_cup_2026_actual_results.csv",
    parse_dates=["date"],
)
world_cup_groups = pd.read_csv(
    project_root / "data" / "world_cup_2026_groups.csv"
)

expected_selected_model = "Calibrated Linear SVM"
if model_metadata["selected_model_name"] != expected_selected_model:
    raise ValueError(
        "The retrospective analysis must use the frozen selected model. "
        f"Expected {expected_selected_model}, found "
        f"{model_metadata['selected_model_name']}."
    )

model_training_end = pd.Timestamp(
    model_metadata["training_end"]
)
model_holdout_end = pd.Timestamp(
    model_metadata["holdout_end"]
)
state_data_end = pd.Timestamp(
    model_metadata["state_data_end"]
)
world_cup_start = world_cup_actual["date"].min()
world_cup_end = world_cup_actual["date"].max()

if state_data_end >= world_cup_start:
    raise ValueError(
        "World Cup result leakage detected in the frozen state."
    )
if len(world_cup_actual) != 104:
    raise ValueError(
        f"Expected 104 World Cup matches, found {len(world_cup_actual)}."
    )
if world_cup_actual["match_id"].nunique() != 104:
    raise ValueError("World Cup match IDs are not unique.")

world_cup_teams = sorted(
    set(world_cup_actual["team_a"])
    | set(world_cup_actual["team_b"])
)

if len(world_cup_teams) != 48:
    raise ValueError(
        f"Expected 48 World Cup teams, found {len(world_cup_teams)}."
    )

unresolved_world_cup_teams = sorted(
    set(world_cup_teams)
    - predictor.available_teams
)

if unresolved_world_cup_teams:
    raise ValueError(
        "World Cup team names missing from the frozen state snapshot: "
        + ", ".join(unresolved_world_cup_teams)
    )

historical_source = pd.read_csv(
    project_root / "data" / "results.csv",
    parse_dates=["date"],
)
completed_world_cup_rows = historical_source.loc[
    historical_source["tournament"].eq("FIFA World Cup")
    & historical_source["date"].between(
        world_cup_start,
        world_cup_end,
        inclusive="both",
    )
    & historical_source[
        ["home_score", "away_score"]
    ].notna().all(axis=1)
]

world_cup_matches_in_model_data = len(
    completed_world_cup_rows
)

if world_cup_matches_in_model_data != 0:
    raise ValueError(
        "Completed World Cup matches entered the frozen historical data."
    )

class_labels = CLASS_LABELS
class_names = CLASS_NAMES
features_upgraded = model_metadata["feature_columns"]

In [ ]:
outcome_to_label = {
    "team_b_win": 0,
    "draw": 1,
    "team_a_win": 2,
}
label_to_outcome = {
    0: "team_b_win",
    1: "draw",
    2: "team_a_win",
}

backtest_rows = []

for match in world_cup_actual.itertuples(index=False):
    prediction = predictor.get_match_probabilities(
        match.team_a,
        match.team_b,
        tournament_weight=1.0,
        neutral=bool(match.neutral),
    )

    probabilities = np.array([
        prediction["away_win"],
        prediction["draw"],
        prediction["home_win"],
    ])

    if not np.isclose(
        probabilities.sum(),
        1.0,
        atol=1e-8,
    ):
        raise ValueError(
            f"Probability sum failed for {match.match_id}."
        )

    predicted_label = int(
        class_labels[np.argmax(probabilities)]
    )
    actual_label = outcome_to_label[
        match.outcome
    ]

    backtest_rows.append({
        "match_id": match.match_id,
        "match_number": int(match.match_number),
        "date": match.date,
        "stage": match.stage,
        "group": match.group,
        "team_a": match.team_a,
        "team_b": match.team_b,
        "neutral": bool(match.neutral),
        "final_score_a": int(match.final_score_a),
        "final_score_b": int(match.final_score_b),
        "decided_by": match.decided_by,
        "advanced_team": match.advanced_team,
        "actual_outcome": match.outcome,
        "actual_label": actual_label,
        "p_team_a_win": prediction["home_win"],
        "p_draw": prediction["draw"],
        "p_team_b_win": prediction["away_win"],
        "predicted_outcome": label_to_outcome[predicted_label],
        "predicted_label": predicted_label,
        "correct_argmax": int(
            predicted_label == actual_label
        ),
    })

world_cup_match_predictions = pd.DataFrame(
    backtest_rows
)

if len(world_cup_match_predictions) != 104:
    raise ValueError(
        "Backtest prediction row count changed unexpectedly."
    )

world_cup_match_predictions.to_csv(
    outputs_dir / "world_cup_2026_match_predictions.csv",
    index=False,
)

actual_labels_wc = world_cup_match_predictions[
    "actual_label"
].to_numpy(dtype=int)
predicted_labels_wc = world_cup_match_predictions[
    "predicted_label"
].to_numpy(dtype=int)
probabilities_wc = world_cup_match_predictions[[
    "p_team_b_win",
    "p_draw",
    "p_team_a_win",
]].to_numpy(dtype=float)


def score_probability_frame(frame):
    y_true = frame["actual_label"].to_numpy(
        dtype=int
    )
    y_pred = frame["predicted_label"].to_numpy(
        dtype=int
    )
    probs = frame[[
        "p_team_b_win",
        "p_draw",
        "p_team_a_win",
    ]].to_numpy(dtype=float)

    recalls = recall_score(
        y_true,
        y_pred,
        labels=class_labels,
        average=None,
        zero_division=0,
    )

    return {
        "matches": len(frame),
        "accuracy": accuracy_score(y_true, y_pred),
        "log_loss": log_loss(
            y_true,
            probs,
            labels=class_labels,
        ),
        "multiclass_brier": multiclass_brier_score(
            y_true,
            probs,
            class_labels,
        ),
        "actual_draw_rate": float(
            (y_true == 1).mean()
        ),
        "mean_predicted_draw_probability": float(
            probs[:, 1].mean()
        ),
        "argmax_draw_prediction_rate": float(
            (y_pred == 1).mean()
        ),
        "argmax_draw_recall": float(recalls[1]),
    }


metric_frames = {
    "Overall": world_cup_match_predictions,
    "Group stage": world_cup_match_predictions.loc[
        world_cup_match_predictions["stage"].eq("group")
    ],
    "Knockout stage": world_cup_match_predictions.loc[
        ~world_cup_match_predictions["stage"].eq("group")
    ],
}

world_cup_match_metrics = pd.DataFrame([
    {
        "split": split_name,
        **score_probability_frame(frame),
    }
    for split_name, frame in metric_frames.items()
])

world_cup_match_metrics.to_csv(
    outputs_dir / "world_cup_2026_match_metrics.csv",
    index=False,
)

baseline_input = pd.DataFrame(
    np.zeros(
        (
            len(world_cup_match_predictions),
            len(features_upgraded),
        )
    ),
    columns=features_upgraded,
)

baseline_probs = align_probability_columns(
    dummy_model,
    dummy_model.predict_proba(baseline_input),
    class_labels,
)
baseline_pred = class_labels[
    np.argmax(
        baseline_probs,
        axis=1,
    )
]

model_vs_baseline = pd.DataFrame([
    {
        "model": model_metadata["selected_model_name"],
        "accuracy": accuracy_score(
            actual_labels_wc,
            predicted_labels_wc,
        ),
        "log_loss": log_loss(
            actual_labels_wc,
            probabilities_wc,
            labels=class_labels,
        ),
        "multiclass_brier": multiclass_brier_score(
            actual_labels_wc,
            probabilities_wc,
            class_labels,
        ),
    },
    {
        "model": "Historical class-prior baseline",
        "accuracy": accuracy_score(
            actual_labels_wc,
            baseline_pred,
        ),
        "log_loss": log_loss(
            actual_labels_wc,
            baseline_probs,
            labels=class_labels,
        ),
        "multiclass_brier": multiclass_brier_score(
            actual_labels_wc,
            baseline_probs,
            class_labels,
        ),
    },
])

model_vs_baseline.to_csv(
    outputs_dir / "world_cup_2026_model_vs_baseline.csv",
    index=False,
)

wc_confusion = confusion_matrix(
    actual_labels_wc,
    predicted_labels_wc,
    labels=class_labels,
)

wc_confusion_rows = []

for actual_index, actual_label in enumerate(
    class_labels
):
    for predicted_index, predicted_label in enumerate(
        class_labels
    ):
        wc_confusion_rows.append({
            "actual_label": int(actual_label),
            "actual_name": class_names[int(actual_label)],
            "predicted_label": int(predicted_label),
            "predicted_name": class_names[int(predicted_label)],
            "count": int(
                wc_confusion[
                    actual_index,
                    predicted_index,
                ]
            ),
        })

pd.DataFrame(
    wc_confusion_rows
).to_csv(
    outputs_dir / "world_cup_2026_confusion_matrix.csv",
    index=False,
)

display(world_cup_match_metrics)
display(model_vs_baseline)

In [ ]:
bootstrap_iterations = 5000
bootstrap_seed = 2026
bootstrap_rng = np.random.default_rng(
    bootstrap_seed
)

bootstrap_accuracy = np.empty(
    bootstrap_iterations
)
bootstrap_log_loss = np.empty(
    bootstrap_iterations
)
bootstrap_brier = np.empty(
    bootstrap_iterations
)

for bootstrap_index in range(
    bootstrap_iterations
):
    sample_index = bootstrap_rng.integers(
        0,
        len(actual_labels_wc),
        size=len(actual_labels_wc),
    )

    sample_y = actual_labels_wc[
        sample_index
    ]
    sample_pred = predicted_labels_wc[
        sample_index
    ]
    sample_probs = probabilities_wc[
        sample_index
    ]

    bootstrap_accuracy[
        bootstrap_index
    ] = accuracy_score(
        sample_y,
        sample_pred,
    )
    bootstrap_log_loss[
        bootstrap_index
    ] = log_loss(
        sample_y,
        sample_probs,
        labels=class_labels,
    )
    bootstrap_brier[
        bootstrap_index
    ] = multiclass_brier_score(
        sample_y,
        sample_probs,
        class_labels,
    )

overall_metrics = world_cup_match_metrics.loc[
    world_cup_match_metrics["split"].eq("Overall")
].iloc[0]

bootstrap_metric_map = {
    "accuracy": (
        float(overall_metrics["accuracy"]),
        bootstrap_accuracy,
    ),
    "log_loss": (
        float(overall_metrics["log_loss"]),
        bootstrap_log_loss,
    ),
    "multiclass_brier": (
        float(overall_metrics["multiclass_brier"]),
        bootstrap_brier,
    ),
}

bootstrap_rows = []

for metric_name, (
    estimate,
    values,
) in bootstrap_metric_map.items():
    lower, upper = np.quantile(
        values,
        [0.025, 0.975],
    )

    bootstrap_rows.append({
        "metric": metric_name,
        "estimate": estimate,
        "ci_lower_95": float(lower),
        "ci_upper_95": float(upper),
        "bootstrap_iterations": bootstrap_iterations,
        "bootstrap_seed": bootstrap_seed,
    })

world_cup_bootstrap_metrics = pd.DataFrame(
    bootstrap_rows
)
world_cup_bootstrap_metrics.to_csv(
    outputs_dir / "world_cup_2026_bootstrap_metrics.csv",
    index=False,
)

figures_dir = outputs_dir / "figures"
figures_dir.mkdir(exist_ok=True)

plot_bootstrap = world_cup_bootstrap_metrics.copy()
x_positions = np.arange(
    len(plot_bootstrap)
)
estimates = plot_bootstrap[
    "estimate"
].to_numpy(dtype=float)

lower_error = (
    estimates
    - plot_bootstrap[
        "ci_lower_95"
    ].to_numpy(dtype=float)
)
upper_error = (
    plot_bootstrap[
        "ci_upper_95"
    ].to_numpy(dtype=float)
    - estimates
)

if (
    np.any(lower_error < 0)
    or np.any(upper_error < 0)
):
    raise ValueError(
        "Bootstrap confidence interval does not contain its point estimate."
    )

plt.figure(figsize=(8, 5))
plt.errorbar(
    x_positions,
    estimates,
    yerr=np.vstack([
        lower_error,
        upper_error,
    ]),
    fmt="o",
    capsize=5,
)
plt.xticks(
    x_positions,
    ["Accuracy", "Log loss", "Brier"],
)
plt.ylabel("Metric value")
plt.title(
    "2026 World Cup backtest with 95% bootstrap intervals"
)
plt.tight_layout()
plt.savefig(
    figures_dir / "world_cup_2026_probability_performance.png",
    dpi=150,
    bbox_inches="tight",
)
plt.close()

display(world_cup_bootstrap_metrics)

In [ ]:
actual_groups = {
    f"Group {group_name}": (
        group_frame
        .sort_values("listed_position")["team"]
        .tolist()
    )
    for group_name, group_frame
    in world_cup_groups.groupby(
        "group",
        sort=True,
    )
}

if len(actual_groups) != 12:
    raise ValueError(
        f"Expected 12 actual groups, found {len(actual_groups)}."
    )
if (
    sum(
        len(teams)
        for teams in actual_groups.values()
    )
    != 48
):
    raise ValueError(
        "Actual group table does not contain exactly 48 team slots."
    )

predictor.precompute_neutral_pairs(world_cup_teams)

post_tournament_simulation_iterations = 2000
post_tournament_simulation_seed = 2026

world_cup_pre_tournament_simulation = (
    run_monte_carlo_tournament(
        predictor,
        actual_groups,
        iterations=post_tournament_simulation_iterations,
        seed=post_tournament_simulation_seed,
    )
)

stage_order = [
    "Group",
    "R32",
    "R16",
    "QF",
    "SF",
    "Final",
    "Champion",
]
stage_numeric = {
    stage: index
    for index, stage in enumerate(
        stage_order
    )
}

actual_stage = {
    team: "Group"
    for team in world_cup_teams
}

stage_to_label = {
    "round_of_32": "R32",
    "round_of_16": "R16",
    "quarter_final": "QF",
    "semi_final": "SF",
    "final": "Final",
}

for stage_name, stage_label in stage_to_label.items():
    stage_matches = world_cup_actual.loc[
        world_cup_actual["stage"].eq(
            stage_name
        )
    ]

    participants = (
        set(stage_matches["team_a"])
        | set(stage_matches["team_b"])
    )

    for team in participants:
        if (
            stage_numeric[stage_label]
            > stage_numeric[actual_stage[team]]
        ):
            actual_stage[team] = stage_label

final_match = world_cup_actual.loc[
    world_cup_actual["stage"].eq("final")
].iloc[0]

actual_champion = final_match["advanced_team"]
actual_runner_up = (
    final_match["team_b"]
    if actual_champion == final_match["team_a"]
    else final_match["team_a"]
)

actual_stage[actual_champion] = "Champion"
actual_stage[actual_runner_up] = "Final"

third_place_match = world_cup_actual.loc[
    world_cup_actual["stage"].eq(
        "third_place"
    )
].iloc[0]

actual_third = third_place_match[
    "advanced_team"
]
actual_fourth = (
    third_place_match["team_b"]
    if actual_third == third_place_match["team_a"]
    else third_place_match["team_a"]
)


def actual_finish_label(team):
    if team == actual_champion:
        return "Champion"
    if team == actual_runner_up:
        return "Runner-up"
    if team == actual_third:
        return "Third"
    if team == actual_fourth:
        return "Fourth"
    return actual_stage[team]


world_cup_expected_vs_actual = (
    world_cup_pre_tournament_simulation.copy()
)

world_cup_expected_vs_actual[
    "actual_stage_reached"
] = (
    world_cup_expected_vs_actual["team"]
    .map(actual_stage)
)

world_cup_expected_vs_actual[
    "actual_finish"
] = (
    world_cup_expected_vs_actual["team"]
    .map(actual_finish_label)
)

world_cup_expected_vs_actual[
    "actual_stage_numeric"
] = (
    world_cup_expected_vs_actual[
        "actual_stage_reached"
    ].map(stage_numeric)
)

world_cup_expected_vs_actual[
    "champion_probability_rank"
] = (
    world_cup_expected_vs_actual[
        "champion_probability"
    ]
    .rank(
        method="min",
        ascending=False,
    )
    .astype(int)
)

world_cup_expected_vs_actual.to_csv(
    outputs_dir / "world_cup_2026_expected_vs_actual.csv",
    index=False,
)

stage_probability_columns = {
    "R32": "r32_probability",
    "R16": "r16_probability",
    "QF": "qf_probability",
    "SF": "sf_probability",
    "Final": "final_probability",
    "Champion": "champion_probability",
}

stage_validation_rows = []

for (
    stage_name,
    probability_column,
) in stage_probability_columns.items():
    actual_binary = (
        world_cup_expected_vs_actual[
            "actual_stage_numeric"
        ]
        >= stage_numeric[stage_name]
    ).astype(int)

    predicted_probability = (
        world_cup_expected_vs_actual[
            probability_column
        ].astype(float)
    )

    slots = int(actual_binary.sum())

    predicted_top_k = set(
        world_cup_expected_vs_actual
        .assign(
            _stage_probability=predicted_probability
        )
        .nlargest(
            slots,
            "_stage_probability",
        )["team"]
    )

    actual_reached = set(
        world_cup_expected_vs_actual.loc[
            actual_binary.eq(1),
            "team",
        ]
    )

    top_k_capture_rate = (
        len(
            predicted_top_k
            & actual_reached
        )
        / slots
        if slots > 0
        else np.nan
    )

    stage_validation_rows.append({
        "stage": stage_name,
        "teams": len(actual_binary),
        "slots": slots,
        "stage_brier_score": float(
            np.mean(
                (
                    predicted_probability
                    - actual_binary
                ) ** 2
            )
        ),
        "top_k_capture_rate": float(
            top_k_capture_rate
        ),
    })

world_cup_stage_validation = pd.DataFrame(
    stage_validation_rows
)
world_cup_stage_validation.to_csv(
    outputs_dir / "world_cup_2026_stage_validation.csv",
    index=False,
)

world_cup_draw_calibration = (
    world_cup_match_predictions
    .assign(
        draw_actual=lambda frame:
            frame["actual_label"]
            .eq(1)
            .astype(int),
        draw_bin=lambda frame:
            pd.qcut(
                frame["p_draw"],
                q=5,
                duplicates="drop",
            ).astype(str),
    )
    .groupby(
        "draw_bin",
        observed=True,
        as_index=False,
    )
    .agg(
        matches=("match_id", "count"),
        mean_predicted_draw_probability=(
            "p_draw",
            "mean",
        ),
        observed_draw_rate=(
            "draw_actual",
            "mean",
        ),
    )
)

world_cup_draw_calibration.to_csv(
    outputs_dir / "world_cup_2026_draw_calibration.csv",
    index=False,
)

plt.figure(figsize=(7, 5))
plt.plot(
    world_cup_draw_calibration[
        "mean_predicted_draw_probability"
    ],
    world_cup_draw_calibration[
        "observed_draw_rate"
    ],
    marker="o",
)
plt.plot(
    [0, 0.5],
    [0, 0.5],
    linestyle="--",
)
plt.xlabel(
    "Mean predicted draw probability"
)
plt.ylabel(
    "Observed draw rate"
)
plt.title(
    "2026 World Cup draw calibration (5 quantile bins)"
)
plt.tight_layout()
plt.savefig(
    figures_dir / "world_cup_2026_draw_calibration.png",
    dpi=150,
    bbox_inches="tight",
)
plt.close()

top_expected = (
    world_cup_expected_vs_actual
    .head(16)
    .copy()
)
top_expected["plot_label"] = (
    top_expected["team"]
    + " — "
    + top_expected["actual_finish"]
)

plt.figure(figsize=(9, 7))
plt.barh(
    top_expected["plot_label"][::-1],
    top_expected[
        "champion_probability"
    ][::-1] * 100,
)
plt.xlabel(
    "Frozen-model champion probability (%)"
)
plt.ylabel(
    "Team and actual finish"
)
plt.title(
    "Actual 2026 field: expected title probability vs actual finish"
)
plt.tight_layout()
plt.savefig(
    figures_dir / "world_cup_2026_expected_vs_actual.png",
    dpi=150,
    bbox_inches="tight",
)
plt.close()

freshness_gap_days = int(
    (
        world_cup_start
        - state_data_end
    ).days
)

backtest_validation = pd.DataFrame([
    ("selected_model", model_metadata["selected_model_name"]),
    ("model_training_end", model_training_end.strftime("%Y-%m-%d")),
    ("historical_holdout_end", model_holdout_end.strftime("%Y-%m-%d")),
    ("pre_tournament_state_data_end", state_data_end.strftime("%Y-%m-%d")),
    ("world_cup_start", world_cup_start.strftime("%Y-%m-%d")),
    ("world_cup_end", world_cup_end.strftime("%Y-%m-%d")),
    ("state_to_world_cup_gap_days", freshness_gap_days),
    ("world_cup_matches_evaluated", len(world_cup_actual)),
    ("world_cup_participants", len(world_cup_teams)),
    ("world_cup_matches_in_model_data", world_cup_matches_in_model_data),
    ("world_cup_outcomes_used_for_retraining", 0),
    ("world_cup_outcomes_used_for_model_reselection", 0),
    ("bootstrap_iterations", bootstrap_iterations),
    ("bootstrap_seed", bootstrap_seed),
    ("actual_field_simulation_iterations", post_tournament_simulation_iterations),
    ("actual_field_simulation_seed", post_tournament_simulation_seed),
], columns=["metric", "value"])

backtest_validation.to_csv(
    outputs_dir / "world_cup_2026_backtest_validation.csv",
    index=False,
)

display(
    world_cup_expected_vs_actual[[
        "team",
        "champion_probability",
        "champion_probability_rank",
        "actual_finish",
    ]].head(16)
)
display(world_cup_stage_validation)
display(backtest_validation)

## Interpretation and limitations

This is a retrospective backtest of a frozen project state, not a claim that the predictions were published prospectively.

The 104-match sample is small enough that point estimates have meaningful sampling uncertainty, so the headline metrics are accompanied by 5,000-resample bootstrap intervals.

The actual-field Monte Carlo comparison uses the real 48-team group field, but it retains the project's approximate knockout mapping, Elo-after-points group tiebreak and neutral-match simulation assumption. It therefore does not reproduce host-country venue advantage for Mexico, Canada or the United States.

The frozen historical state ends 72 days before the opening match, so later friendlies, qualifiers, final squad selections, injuries and tactical changes are deliberately excluded.